# Getting started with MemsArray objects

The `MemsArray` class is the base class for all `Megamicros` antennas. 
Used on its own, it doesn't do anything. This Notebook shows how it can be used with the right tools.

In [1]:
import numpy as np
from megamicros.log import log
from megamicros.core.base import MemsArray

log.setLevel( "INFO" )

## Without any initialization

The ``MemsArray`` constructor allows the user to completely define an antenna. 
Without arguments, the user will have to do it himself by using the class methods.

In [ ]:
# Define an empty antenna
antenna = MemsArray()

# Init the antenna
antenna.setAvailableMems( 10 )
antenna.setActiveMems( [0,1,2])


## With initialization parameters 

In [ ]:
# Create a 10 MEMs antenna
antenna = MemsArray( available_mems_number=10, mems=[0, 1, 2] )

## With antenna physical parameters

In [2]:
# Create an antenna from physical parameters
antenna_definition = np.load ('Antenna-square-JetsonNano-0001.npy', allow_pickle=True )
mems_position = antenna_definition.item().get('positions')
antenna = MemsArray( mems_position=mems_position )

# Create antenna and set activated MEMs 
antenna = MemsArray( 
    available_mems_number = len( antenna_definition.item().get('available_mems') ),
    mems_position=mems_position,
    mems=antenna_definition.item().get('mems') 
)



2023-09-13 16:05:55,660 [INFO]:  .Setting available MEMs numbered from 0 to 24
2023-09-13 16:05:55,663 [INFO]:  .Set a 25 activable MEMs antenna with physical positions
2023-09-13 16:05:55,665 [INFO]:  .Created a new antenna
2023-09-13 16:05:55,666 [INFO]:  .Set a 32 MEMs antenna with MEMs numbered from 0 to 31
2023-09-13 16:05:55,667 [INFO]:  .Set a 25 activable MEMs antenna with physical positions
2023-09-13 16:05:55,669 [INFO]:  .25 MEMs were activated among 0 to 31 available MEMs
2023-09-13 16:05:55,671 [INFO]:  .Created a new antenna


In [4]:
antenna.setActiveMems([0, 1, 31])

2023-09-13 16:06:07,198 [WARNING]: in megamicros.log (base.py:277): MEMs are already activated with physical positions set. Please consider that new activated map could not match with physical MEMs positions
2023-09-13 16:06:07,201 [INFO]:  .3 MEMs were activated among 0 to 31 available MEMs


## Getting signal from DB

In [ ]:
# Available labels
LABEL_SOW_FEEDING_CALL = 18
LABEL_PIGLET_SQUEALS = 15
LABEL_SOW_GRUNT_NERVOUS = 16
LABEL_ROOM_NOISE = 29
LABEL_SOW_GRUNT = 8
LABEL_SOW_GRUNT_MODSTRESS  = 1
LABEL_SOW_SCREAMS = 3
LABEL_PIGLET_SQUEALS_2 = 5

# choose label, file and sequence in file:
LABEL_ID = LABEL_SOW_FEEDING_CALL
FILE_ID = 8692          # 5838 (1), 7135 (1), 6860(3), 6560(1)
SEQUENCE_ID = 0         

with AidbSession(
    dbhost='http://dbwelfare.biimea.io/',
    login='ailab',
    email='bruno.gas@biimea.com',
    password='#T;uZnQ5UJ_JC~&' ) as session:
        signal: MuAudio = session.load_labelized( 
            sourcefile_id=FILE_ID, 
            label_id=LABEL_ID, 
            limit=100, 
            channels=list( np.arange( 32 ) + 1 ) 
        )[SEQUENCE_ID]

# get infos
LABEL_TXT = signal.label
CHANNELS_NUMBER = signal.channels_number
SAMPLES_NUMBER = signal.samples_number
SAMPLING_FREQUENCY = signal.sampling_frequency

print( f"Some informations about the signal loaded:" )
print( f" > label={LABEL_TXT}" )
print( f" > channels_number={CHANNELS_NUMBER}" )
print( f" > samples_number={SAMPLES_NUMBER}" )
print( f" > sampling_frequency={SAMPLING_FREQUENCY}" )

In [ ]:
# Play sound using channel 0 and 1
left = np.array( signal.channel(0) )
right = np.array( signal.channel(1) )
sound = np.array( [left, right] ).T

display.Audio( sound, rate=SAMPLING_FREQUENCY )

## Set the beamformer

In [ ]:
FRAME_LENGTH = 1024
AREA = [12, 14, 0.01]
AREA_QUANTIZATION = [4, 4, 1/0.01]

# Get the antenna physical description
antenna = np.load ('Antenna-square-JetsonNano-0001.npy', allow_pickle=True )
mems_position = antenna.item().get("positions")

# Create the beamformer
bmf = Beamformer( 
    mems_position = mems_position,
    sampling_frequency = SAMPLING_FREQUENCY,
    window_size = FRAME_LENGTH,    
    area = AREA,
    area_quantization = AREA_QUANTIZATION
)

# Move the antenna in the right place:
bmf.moveArea( [0, 0, -2] )

# Limit the frequency bandwidth for BF computing
bmf.setBandWidth( [200, 2000], unit="frequency" )

# Init the beamformer
bmf.init()

# print area locations and antenna 
space_locations = bmf.getLocations()
mems_location = bmf.getMems()
nx, ny, nz = bmf.getLocationsNumber()

fig = plt.figure()
ax = fig.add_subplot( 121, projection='3d' )
ax.scatter( space_locations[:,0], space_locations[:,1], space_locations[:,2] )
ax.scatter( mems_location[:,0], mems_location[:,1], mems_location[:,2], marker='^' )
ax = fig.add_subplot( 122, projection='3d' )
ax.scatter( mems_location[:,0], mems_location[:,1], mems_location[:,2], marker='^' )
fig.show()


## Compute preformed channels 

In [ ]:
# Get the whole 32 channels signal as a numpy.ndarray
signal32 = signal().T

# Check if some available mems have not been activated
# Remove from signal if any
mems = antenna.item().get('mems')
available_mems = antenna.item().get('available_mems')
if False in np.isin( available_mems, mems ):
    mask = list( np.invert( np.logical_not( np.isin( available_mems, mems ) ) ) )
    signal32 = signal32[:,mask]

FRAMES_NUMBER = SAMPLES_NUMBER // FRAME_LENGTH - 1

print( f"{FRAMES_NUMBER} frames of {FRAME_LENGTH} samples to perform... " )


imgs = []
for i in range( FRAMES_NUMBER ):
    bf = bmf.beamform( signal32[i*FRAME_LENGTH:(i+1)*FRAME_LENGTH,:] )
    imgs.append( np.reshape( bf, (nx, ny) ) )

### Make video

In [ ]:
generate_moovie( 
    imgs, 
    rate=SAMPLING_FREQUENCY/FRAME_LENGTH,
    sound=sound.astype( np.float32 ).T,
    sampling_frequency=SAMPLING_FREQUENCY,
    norm=None,
    extent=( 0, AREA[0], 0, AREA[1] ),
    cleanup=True
)

### Do the same with an antenna object

In [ ]:
# Declare a MEMs antenna
antenna = MemsArray( available_mems_number=CHANNELS_NUMBER )

# set active mems
antenna.setActiveMems( [i for i in range( CHANNELS_NUMBER )] )
print( f"active mems number={antenna.mems_number}" )

# iterate over the antenna data stream
#for i, data in enumerate( antenna ):
#    print( f"data={data}")
#    if i > 10:
#        break
